# Activity — LLMOps & Monitoring

## Lesson 14 — LLMOps & Monitoring

In this activity, we will add basic monitoring and evaluation to an existing AI agent.

We will learn how to track:

- Logs
- Latency
- Cost
- Tracing
- Prompt versions
- Model versions
- Evaluation results

The goal is to understand how an AI application can be monitored after it has been built.

## Learning Objectives

By the end of this activity, you will be able to:

- Understand the basic idea of LLMOps
- Add logging to an AI agent
- Measure response latency
- Track basic token usage and estimated cost
- Add simple tracing
- Manage prompt versions
- Manage model versions
- Create a simple evaluation dataset
- Evaluate an agent's responses
- Combine monitoring and evaluation into one workflow

## What is LLMOps?

LLMOps means managing and monitoring applications that use Large Language Models (LLMs).

An AI application needs more than just a working model.

We also need to know:

- How long does a response take?
- How many tokens are being used?
- How much does each request cost?
- Which model version is being used?
- Which prompt version is being used?
- Is the agent producing good responses?
- What happened when an error occurred?

This activity introduces a simple version of these practices.

## Basic LLMOps Flow

```text
User
  ↓
Agent
  ↓
LLM
  ↓
Response
  ↓
Monitoring
 ├── Logs
 ├── Latency
 ├── Cost
 ├── Tracing
 └── Evaluation


## Step 1 — Install the Required Library

We only need the Google GenAI SDK for this activity.

We will use Python's built-in libraries for monitoring.

In [ ]:
!pip -q install google-genai

## Step 2 — Import Libraries

In [ ]:
import os
import time
import getpass

## Step 3 — Add Your Gemini API Key

We will store the API key in an environment variable instead of writing it directly in the code.

In [ ]:
api_key = getpass.getpass("Enter your Gemini API key: ")

os.environ["GEMINI_API_KEY"] = api_key

print("API key loaded successfully.")

## Step 4 — Create the Gemini Client

In [ ]:
from google import genai

client = genai.Client(
    api_key=os.environ["GEMINI_API_KEY"]
)

print("Gemini client created.")

## Step 5 — Model Management

Model management means keeping track of which model our application is using.

For this activity, we will store the model name in one variable.

In [ ]:
MODEL = "gemini-3.8-flash"

print("Model:", MODEL)

## Step 6 — Prompt Management

Prompts are also part of an AI application's configuration.

Instead of writing the prompt directly inside the agent function, we will store it separately.

We will also give the prompt a version number.

In [ ]:
PROMPT_VERSION = "v1"

SYSTEM_PROMPT = """
You are a helpful AI assistant.
Give clear and concise answers.
"""

print("Prompt version:", PROMPT_VERSION)

## Step 7 — Create a Simple Agent

This is the existing agent that we will monitor.

The agent receives a user message and sends it to the LLM.

In [ ]:
def run_agent(user_message):

    prompt = f"""
{SYSTEM_PROMPT}

User:
{user_message}
"""

    response = client.models.generate_content(
        model=MODEL,
        contents=prompt
    )

    return response

## Step 8 — Test the Agent

In [ ]:
response = run_agent(
    "What is an AI agent?"
)

print(response.text)


## Step 9 — Add Logging

Logging means recording important information about what happened during a request.

For example:

- User request
- Model used
- Prompt version
- Response
- Request status

Logs help developers understand what happened when an application is running.

In [ ]:
def log_event(message):
    print(f"[LOG] {message}")

## Step 10 — Test Logging

In [ ]:
log_event("Agent started")
log_event(f"Model: {MODEL}")
log_event(f"Prompt version: {PROMPT_VERSION}")
log_event("Agent completed")

## Step 11 — Measure Latency

Latency is the amount of time an application takes to produce a response.

We can measure it by recording the start and end time of a request.

In [ ]:
start_time = time.time()

response = run_agent(
    "Explain machine learning in one sentence."
)

end_time = time.time()

latency = end_time - start_time

print("Latency:", round(latency, 2), "seconds")
print("Response:", response.text)

## Step 12 — Track Token Usage

LLM applications often charge based on token usage.

The API response may provide usage information.

We will inspect the usage information returned by the model.

In [ ]:
print(response.usage_metadata)

## Step 13 — Estimate Cost

For learning purposes, we will create a simple cost calculation.

The actual price depends on the model and provider.

The value below is only an example rate for demonstrating the calculation.

In [ ]:
input_tokens = response.usage_metadata.prompt_token_count
output_tokens = response.usage_metadata.candidates_token_count

INPUT_COST_PER_1K = 0.001
OUTPUT_COST_PER_1K = 0.002

estimated_cost = (
    (input_tokens / 1000) * INPUT_COST_PER_1K
    +
    (output_tokens / 1000) * OUTPUT_COST_PER_1K
)

print("Input tokens:", input_tokens)
print("Output tokens:", output_tokens)
print("Estimated cost: $", round(estimated_cost, 6))

## Step 14 — Basic Tracing

Tracing means following the steps of a request through an application.

For our simple agent, we can trace:

```text
Request
   ↓
Prompt
   ↓
LLM
   ↓
Response
   ↓
Monitoring

We will create a simple trace ID for each request.


In [ ]:
import uuid

trace_id = str(uuid.uuid4())

print("Trace ID:", trace_id)

## Step 15 — Create a Monitored Agent

Now we will combine:

- Logging
- Latency
- Cost tracking
- Tracing
- Model version
- Prompt version

into one function.

In [ ]:
def monitored_agent(user_message):

    trace_id = str(uuid.uuid4())

    log_event(f"Trace ID: {trace_id}")
    log_event(f"Model: {MODEL}")
    log_event(f"Prompt version: {PROMPT_VERSION}")

    start_time = time.time()

    try:

        response = run_agent(user_message)

        end_time = time.out()

        latency = end_time - start_time

        input_tokens = response.usage_metadata.prompt_token_count
        output_tokens = response.usage_metadata.candidates_token_count

        estimated_cost = (
            (input_tokens / 1000) * INPUT_COST_PER_1K
            +
            (output_tokens / 1000) * OUTPUT_COST_PER_1K
        )

        log_event("Request completed")
        log_event(f"Latency: {round(latency, 2)} seconds")
        log_event(f"Input tokens: {input_tokens}")
        log_event(f"Output tokens: {output_tokens}")
        log_event(f"Estimated cost: ${round(estimated_cost, 6)}")

        return response

    except Exception as e:

        log_event(f"Request failed: {e}")

        raise

## Step 16 — Test the Monitored Agent

In [ ]:
response = monitored_agent(
    "What is the purpose of an API?"
)

print("\nFinal response:")
print(response.text)

## Step 17 — What is Evaluation?

Evaluation means checking whether an AI application is producing useful responses.

For example, we can evaluate:

- Correctness
- Relevance
- Quality
- Safety
- Response format

In this activity, we will use a simple rule-based evaluation approach.

## Step 18 — Create an Evaluation Dataset

An evaluation dataset contains test questions and expected concepts.

We will keep it small so that the activity remains lightweight.

In [ ]:
evaluation_dataset = [
    {
        "question": "What is an API?",
        "expected_keyword": "interface"
    },
    {
        "question": "What is machine learning?",
        "expected_keyword": "learning"
    },
    {
        "question": "What is an AI agent?",
        "expected_keyword": "agent"
    }
]

print("Evaluation examples:", len(evaluation_dataset))

## Step 19 — Create an Evaluation Function

Our simple evaluator will check whether the expected keyword appears in the model's response.

This is not a complete evaluation system.

It is a simple example to demonstrate the idea of an evaluation pipeline.

In [ ]:
def evaluate_response(response_text, expected_keyword):

    response_text = response_text.lower()

    if expected_keyword.lower() in response_text:
        return "PASS"

    return "FAIL"

## Step 20 — Test the Evaluation Function

In [ ]:
result = evaluate_response(
    "An API is an interface that allows applications to communicate.",
    "interface"
)

print("Evaluation result:", result)

## Step 21 — Build a Simple Evaluation Pipeline

Now we will run all test questions through the agent.

For each question, we will:

1. Send the question to the agent
2. Receive the response
3. Evaluate the response
4. Store the result

In [ ]:
evaluation_results = []

for item in evaluation_dataset:

    response = monitored_agent(
        item["question"]
    )

    result = evaluate_response(
        response.text,
        item["expected_keyword"]
    )

    evaluation_results.append({
        "question": item["question"],
        "result": result
    })

print("Evaluation completed.")

## Step 22 — View Evaluation Results

In [ ]:
for item in evaluation_results:

    print("Question:", item["question"])
    print("Result:", item["result"])
    """ * 40)

## Step 23 — Calculate Evaluation Score

We can calculate a simple pass rate.

```text
Evaluation Score =
Number of Passed Tests / Total Tests


In [ ]:
passed = 0

for item in evaluation_results:

    if item["result"] == "PASS":
        passed += 1

total = len(evaluation_results)

score = passed / total

print("Passed:", passed)
print("Total:", total)
print("Evaluation score:", round(score * 100, 2), "%")

## Step 24 — Create a Monitoring Summary

We have now implemented a lightweight monitoring system.

Our agent can track:

| Feature | What we track |
|---|---|
| Logging | Important application events |
| Latency | Response time |
| Cost | Estimated token cost |
| Tracing | Trace ID for requests |
| Prompt Management | Prompt version |
| Model Management | Model version |
| Evaluation | Pass/fail results |

## Complete LLMOps Flow

Our final system looks like this:

```text
User
  ↓
Monitored Agent
  ↓
Prompt + Model
  ↓
LLM
  ↓
Response
  ↓
┌─────────────────────┐
│ Monitoring          │
│                     │
│ Logs                │
│ Latency             │
│ Cost                │
│ Trace ID            │
└─────────────────────┘
  ↓
Evaluation
  ↓
PASS / FAIL


## Key Takeaways

In this activity, we learned that building an AI agent is only one part of an AI application.

After deployment, we also need to monitor the application.

Important LLMOps practices include:

- Logging application events
- Monitoring latency
- Tracking token usage and cost
- Tracing requests
- Managing prompts
- Managing model versions
- Evaluating responses
- Running evaluation tests regularly

These practices help developers understand and improve AI applications over time.

# Activity Complete

You have built a simple LLMOps and monitoring layer around an AI agent.

You implemented:

- Logging
- Latency monitoring
- Cost tracking
- Tracing
- Prompt versioning
- Model management
- Evaluation
- Evaluation scoring